In [29]:
from jinja2 import Template
import pandas as pd
import yaml
import sys
from itertools import product
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
import outlines
from outlines import Generator
from tqdm import tqdm
sys.path.append("../")
from src.utils import list_to_str
device = "cpu"

In [2]:
sjt_core_set = pd.read_csv("SJT_Core_Data_Set.csv")

with open('../configs/sjt_seeds.yaml', 'r') as file:
    sjt_seeds = yaml.safe_load(file)
    
with open('../configs/personas_v2.yaml', 'r') as file:
    personas = yaml.safe_load(file)

In [43]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=t.float16
).to()

In [3]:
all_seeds = {
    "time_of_day" : sjt_seeds['time_of_day'],
    "suspect1_race": sjt_seeds['suspect_race'],
    "suspect2_race": sjt_seeds['suspect_race'],
    "suspect3_race": sjt_seeds['suspect_race'],
    "suspect1_age": sjt_seeds['suspect_age'],
    "suspect2_age": sjt_seeds['suspect_age'],
    "suspect3_age": sjt_seeds['suspect_age'],
    "suspect1_gender": sjt_seeds['suspect_gender'],
    "suspect2_gender": sjt_seeds['suspect_gender'],
    "suspect3_gender": sjt_seeds['suspect_gender'],
    "officer1_gender": sjt_seeds['officer_gender'],
    "officer2_gender": sjt_seeds['officer_gender'],
    "officer1_age": sjt_seeds['officer_age'],
    "officer2_age": sjt_seeds['officer_age'],
    "officer1_race": sjt_seeds['officer_race'],
    "officer2_race": sjt_seeds['officer_race'],
}

In [4]:
def expand_dict_combinations(data):
    keys = list(data.keys())
    values = [data[k] for k in keys]
    
    combos = []
    for prod in product(*values):
        combos.append(dict(zip(keys, prod)))
    return combos

In [5]:
sjt_seed_all_combos = expand_dict_combinations(all_seeds)

In [41]:
sjt_template_set = []
sjt_full_set = []
for index, row_series in sjt_core_set.iterrows():
    question = row_series['Question'].replace("[","{{ ").replace("]"," }}")
    question_template = Template(question)
    row_dict = row_series.to_dict()
    row_dict['Question'] = question
    sjt_template_set.append(row_dict)
    for config in sjt_seed_all_combos[:10000]:
        rendered_question = question_template.render(config)
        row_dict = row_series.to_dict()
        row_dict['Question'] = rendered_question
        sjt_full_set.append(row_dict)

In [42]:
sjt_df = pd.DataFrame(sjt_full_set).drop_duplicates()
print(sjt_df.shape)

(427, 13)


In [44]:
pd.DataFrame(sjt_template_set).to_csv("sjt_template_set.csv", index = False)
sjt_df.to_csv("augmented_sjt_df.csv", index = False)

In [45]:
answer_indexes = ['1','2','3','4','5','6']
answer_cols = ['Option 1', 'Option 2', 'Option 3', 'Option 4', 'Option 5', 'Option 6']

In [46]:
base_text = personas['law_enforcement']['base_text']
persona = personas['law_enforcement']['personas'][0]

In [47]:
persona_hexaco_template = outlines.Template.from_string("""
<|im_start>user
{{base_text}} with following attributes :

{{attributes}}

Task: Answer the below multiple choice questions:

{{ question }}

Choices for these questions are:

{{ answer_options }}

Answer the question using one of the values from {{ answer_indexes }}.
<|im_end>
<|im_start>assistant
""")

In [48]:
print(persona_hexaco_template(question = sjt_df['Question'][0],
                        answer_indexes = answer_indexes,
                        base_text = base_text,
                        attributes = ",\n".join([f"{key} : {persona[key]}" for key in persona]),
                        answer_options = list_to_str(sjt_df[answer_cols].iloc[0].values) ))

<|im_start>user
You are a law enforcement officer. with following attributes :

Name : John (Jack) Singleton (Preferred Name - Jack),
Age : 46 years,
Presenting Problems : Veteran homicide detective who faces chronic stress, moral injury, and mild social withdrawal but stays highly functional in his demanding work.,
Medical History : No major medical issues apart from stress-related hypertension, occasional migraines, and short-term sleep aid use.,
Family History : Raised in a working-class Watts family marked by alcohol use and street violence, lost a brother to gang violence, fueling his commitment to policing.,
Educational History : Finished high school, joined the force at 21, and has spent 15 years in homicide, deeply tied to his community and meticulous in his casework.,
Emotional Functioning : Copes with intrusive memories and emotional detachment through moderate drinking and work focus but struggles with marital strain and guarded parenting.,
Social Functioning : Mostly social

In [50]:
job_title = 'law_enforcement'
answers = []
sjt_prompts_handmade_personas = []
for index, row in sjt_df.iterrows():
    base_text = personas[job_title]['base_text']
    persona_list = personas[job_title]['personas']
    for persona in tqdm(persona_list, desc="Personas", position=0):
        prompt = persona_hexaco_template(question = row['Question'],
                            answer_indexes = answer_indexes,
                            base_text = base_text,
                            attributes = ",\n".join([f"{key} : {persona[key]}" for key in persona]),
                            answer_options = list_to_str(row[answer_cols].values))
        sjt_prompts_handmade_personas.append(prompt)

Personas: 100%|██████████| 4/4 [00:00<00:00, 5082.46it/s]
